# Phase 5 — Quality-Aware Ensemble Framework (RQ4)

**Objective:** Tie everything together into a single inference pipeline:
1. A *quality classifier* (trained on EyeQ) predicts incoming image quality (`good` / `usable` / `reject`).
2. A *router* picks the optimal **enhancement → model → XAI** combination.
3. A *Clinical Trust Score* combines accuracy with XAI fidelity per prediction.

Routing rules used here (modify in the `ROUTES` dict if you want to test alternatives):

| Quality | Enhancement | Model        | XAI       |
|---------|-------------|--------------|-----------|
| good    | none        | best_clean   | best_clean|
| usable  | CLAHE       | best_clean   | best_clean|
| reject  | GenAI       | best_robust  | best_robust|

`best_clean` and `best_robust` are chosen automatically from the Phase 2 / Phase 4 results (highest mean accuracy on clean and on degraded respectively).

**Outputs (under `Drive/Thesis/results/phase5_quality_ensemble/`):**
- `metrics/quality_classifier_metrics.json`
- `metrics/ensemble_predictions.csv` — per-image route + prediction + trust score
- `metrics/ensemble_summary.csv`     — aggregate vs single-best baseline
- `plots/trust_score_distribution.png`, `confusion_matrix.png`
- `samples/route_<id>.png` — qualitative figure showing the chosen route + heatmap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q timm scikit-learn

## Config

In [ ]:
from pathlib import Path
import torch, json
import numpy as np
import pandas as pd

DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
PHASE_DIR       = RESULTS_ROOT / 'phase5_quality_ensemble'
PHASE1_DIR      = RESULTS_ROOT / 'phase1_data_engineering'
PHASE2_DIR      = RESULTS_ROOT / 'phase2_model_benchmarking'
PHASE4_DIR      = RESULTS_ROOT / 'phase4_genai_enhancement'
for sub in ('metrics', 'plots', 'samples', 'logs'):
    (PHASE_DIR / sub).mkdir(parents=True, exist_ok=True)

LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
EYEQ_DIR     = LOCAL_ROOT / 'eyeq'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
ENHANCED_DIR = LOCAL_ROOT / 'enhanced'

APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('train_images') if p.is_dir())
PRISTINE_CSV = PHASE1_DIR / 'metrics' / 'pristine_split.csv'
EYEQ_CSV     = next(EYEQ_DIR.rglob('*.csv'), None)

NUM_CLASSES = 5
IMAGE_SIZE  = 224
MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50', 'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}
EYEQ_QUALITY_LABELS = {0: 'good', 1: 'usable', 2: 'reject'}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 1. Train (or load) the quality classifier on EyeQ

A small ResNet-18 head — three classes (good/usable/reject). Fast: ~1 epoch is usually enough on EyeQ.

In [ ]:
import timm
import torch.nn as nn
from PIL import Image
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import classification_report, accuracy_score

TFM_EVAL = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
                       T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
TFM_TRAIN = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                        T.RandomHorizontalFlip(),
                        T.ToTensor(),
                        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class EyeQDataset(Dataset):
    """Expects EYEQ_CSV with columns [image, quality] and EYEQ images on disk."""
    def __init__(self, csv_path, images_root, transform,
                 id_col='image', quality_col='quality'):
        self.df = pd.read_csv(csv_path)
        self.df[id_col] = self.df[id_col].astype(str)
        self.images_root = Path(images_root)
        self.transform = transform
        self.id_col = id_col; self.quality_col = quality_col
        # Keep only rows whose image we can find (recursive).
        all_imgs = {p.stem: p for p in self.images_root.rglob('*.*')
                    if p.suffix.lower() in ('.png', '.jpg', '.jpeg')}
        self.df['__path'] = self.df[id_col].apply(lambda s: all_imgs.get(Path(s).stem))
        self.df = self.df.dropna(subset=['__path']).reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['__path']).convert('RGB')
        return self.transform(img), int(row[self.quality_col])

def build_quality_model():
    return timm.create_model('resnet18', pretrained=True, num_classes=3)

Q_CKPT = CHECKPOINTS_DIR / 'quality_classifier.pt'

def train_quality_classifier(epochs=2, batch_size=32):
    if EYEQ_CSV is None:
        print('No EyeQ CSV — quality classifier will be skipped.')
        return None, None
    ds = EyeQDataset(EYEQ_CSV, EYEQ_DIR, TFM_TRAIN)
    n_val = max(1, int(0.15 * len(ds)))
    train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val],
                                     generator=torch.Generator().manual_seed(0))
    val_ds.dataset = EyeQDataset(EYEQ_CSV, EYEQ_DIR, TFM_EVAL)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    vl = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2)
    model = build_quality_model().to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    crit  = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optim.zero_grad(); loss = crit(model(x), y); loss.backward(); optim.step()
        # eval
        model.eval(); ys, ps = [], []
        with torch.no_grad():
            for x, y in vl:
                p = model(x.to(DEVICE)).argmax(1).cpu().numpy()
                ps.append(p); ys.append(y.numpy())
        y = np.concatenate(ys); p = np.concatenate(ps)
        print(f'epoch {ep+1}: val acc = {accuracy_score(y, p):.4f}')
    rep = classification_report(y, p, target_names=list(EYEQ_QUALITY_LABELS.values()), output_dict=True, zero_division=0)
    torch.save(model.state_dict(), Q_CKPT)
    with open(PHASE_DIR / 'metrics' / 'quality_classifier_metrics.json', 'w') as f:
        json.dump(rep, f, indent=2)
    return model, rep

if Q_CKPT.exists():
    qmodel = build_quality_model().to(DEVICE)
    qmodel.load_state_dict(torch.load(Q_CKPT, map_location=DEVICE)); qmodel.eval()
    print('Loaded existing quality classifier.')
else:
    qmodel, _ = train_quality_classifier()

## 2. Pick `best_clean` and `best_robust` models from prior results

In [ ]:
stress_df = pd.read_csv(PHASE2_DIR / 'metrics' / 'stress_test_results.csv')
best_clean  = (stress_df[stress_df['degradation'] == 'clean']
                .sort_values('accuracy', ascending=False).iloc[0]['model'])
best_robust = (stress_df[stress_df['degradation'] != 'clean']
                .groupby('model')['accuracy'].mean().idxmax())
print('best_clean :', best_clean)
print('best_robust:', best_robust)

ROUTES = {
    'good':   {'enhancement': 'none',  'model': best_clean,  'xai': 'gradcam' if best_clean  != 'vit_base' else 'attention'},
    'usable': {'enhancement': 'clahe', 'model': best_clean,  'xai': 'gradcam' if best_clean  != 'vit_base' else 'attention'},
    'reject': {'enhancement': 'genai', 'model': best_robust, 'xai': 'gradcam' if best_robust != 'vit_base' else 'attention'},
}
ROUTES

## 3. Load classifier checkpoints + enhancers

In [ ]:
import cv2
_clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
def enhance_clahe(img):
    arr = np.array(img.convert('RGB'))
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    lab[..., 0] = _clahe.apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

def build_classifier(name):
    m = timm.create_model(TIMM_NAMES[name], pretrained=False, num_classes=NUM_CLASSES)
    m._dr_arch = name; return m

def load_classifier(name):
    m = build_classifier(name).to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE)
    m.load_state_dict(state['state_dict']); m.eval()
    return m

classifiers = {n: load_classifier(n) for n in {best_clean, best_robust}}
print('Loaded classifiers:', list(classifiers))

# GenAI processor: prefer pre-built enhanced sets from Phase 4 (faster).
PHASE4_GENAI_DIR = ENHANCED_DIR / 'genai'
USE_PRECOMPUTED_GENAI = PHASE4_GENAI_DIR.exists()
print('Using precomputed GenAI from Phase 4:', USE_PRECOMPUTED_GENAI)

## 4. End-to-end inference function

In [ ]:
import torch.nn.functional as F

def predict_quality(img: Image.Image) -> str:
    if qmodel is None: return 'good'
    x = TFM_EVAL(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        idx = int(qmodel(x).argmax(1).item())
    return EYEQ_QUALITY_LABELS[idx]

def apply_enhancement(img: Image.Image, kind: str, kind_hint=None, level_hint=None, id_hint=None) -> Image.Image:
    if kind == 'none':  return img
    if kind == 'clahe': return enhance_clahe(img)
    if kind == 'genai':
        if USE_PRECOMPUTED_GENAI and kind_hint and level_hint and id_hint:
            cached = PHASE4_GENAI_DIR / kind_hint / level_hint / f'{id_hint}.png'
            if cached.exists(): return Image.open(cached).convert('RGB')
        return enhance_clahe(img)  # last-resort fallback
    return img

def get_xai_target_layer(model):
    a = model._dr_arch
    return {'resnet50': model.layer4[-1], 'efficientnet_b3': model.blocks[-1],
            'vit_base': model.blocks[-1].norm1}[a]

def gradcam_heatmap(model, x, target_class):
    layer = get_xai_target_layer(model)
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda _, __, o: acts.update(v=o.detach()))
    h2 = layer.register_full_backward_hook(lambda _, gi, go: grads.update(v=go[0].detach()))
    try:
        model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        logits = model(x); logits[0, target_class].backward()
        a, g = acts['v'][0], grads['v'][0]
        cam = torch.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
        cam -= cam.min()
        if cam.max() > 0: cam /= cam.max()
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)
        return cam.squeeze().detach().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

@torch.no_grad()
def attention_rollout(model, x):
    attns, handles = [], []
    for blk in model.blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try: _ = model(x)
    finally:
        for h in handles: h.remove()
    res = None
    for a in attns:
        a = a.mean(1) + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(-1, keepdim=True)
        res = a if res is None else a @ res
    cls = res[0, 0, 1:]; side = int(cls.numel() ** 0.5)
    h = F.interpolate(cls.reshape(side, side)[None, None],
                       size=x.shape[-2:], mode='bilinear', align_corners=False).squeeze().cpu().numpy()
    if h.max() > h.min(): h = (h - h.min()) / (h.max() - h.min())
    return h

XAI_FNS = {'gradcam': gradcam_heatmap, 'attention': attention_rollout}

@torch.no_grad()
def insertion_auc(model, x, heatmap, target_class, steps=20):
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    base = F.avg_pool2d(x, 15, stride=1, padding=7)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = base.clone(); x_mod[0, :, rows, cols] = x[0, :, rows, cols]
        probs.append(torch.softmax(model(x_mod), 1)[0, target_class].item())
    return float(np.trapz(probs, dx=1 / steps))

def trust_score(prediction_prob, insertion):
    """Trust = 0.5 * model confidence + 0.5 * normalised insertion AUC."""
    return float(0.5 * prediction_prob + 0.5 * insertion)

def pipeline(img: Image.Image, kind_hint=None, level_hint=None, id_hint=None):
    quality = predict_quality(img)
    route   = ROUTES[quality]
    enhanced = apply_enhancement(img, route['enhancement'],
                                  kind_hint=kind_hint, level_hint=level_hint, id_hint=id_hint)
    model = classifiers[route['model']]
    x = TFM_EVAL(enhanced).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(model(x), 1)[0]
        pred = int(prob.argmax().item()); conf = float(prob[pred].item())
    heat  = XAI_FNS[route['xai']](model, x, pred)
    ins   = insertion_auc(model, x, heat, pred)
    return {'quality_pred': quality, 'route': route, 'pred': pred, 'conf': conf,
            'insertion_auc': ins, 'trust': trust_score(conf, ins), 'heatmap': heat,
            'enhanced_img': enhanced}

## 5. Run pipeline on the held-out test set

For every test id we evaluate three conditions: clean, blur-high, exposure-high. (Add more if you have time — they're appended to the same CSV.)

In [ ]:
from tqdm.auto import tqdm

test_ids  = pd.read_csv(PHASE2_DIR / 'metrics' / 'test_ids.csv')['id_code'].astype(str).tolist()
labels_df = pd.read_csv(PRISTINE_CSV).set_index('id_code')['diagnosis']

EVAL_CONDITIONS = [
    ('clean',    None, None),
    ('blur',     'blur', 'high'),
    ('exposure', 'exposure', 'high'),
]

rows = []
for id_code in tqdm(test_ids[:80], desc='ensemble'):  # cap at 80 for time
    if id_code not in labels_df.index: continue
    label = int(labels_df[id_code])
    for tag, kind, level in EVAL_CONDITIONS:
        src = APTOS_IMAGES / f'{id_code}.png' if tag == 'clean' \
              else DEGRADED_DIR / kind / level / f'{id_code}.png'
        if not src.exists(): continue
        img = Image.open(src).convert('RGB')
        out = pipeline(img, kind_hint=kind, level_hint=level, id_hint=id_code)
        rows.append({'id_code': id_code, 'condition': tag,
                     'true_label': label, 'pred': out['pred'], 'correct': int(out['pred'] == label),
                     'quality_pred': out['quality_pred'], 'route_model': out['route']['model'],
                     'route_enhancement': out['route']['enhancement'],
                     'route_xai': out['route']['xai'], 'conf': out['conf'],
                     'insertion_auc': out['insertion_auc'], 'trust': out['trust']})

ens_df = pd.DataFrame(rows)
ens_csv = PHASE_DIR / 'metrics' / 'ensemble_predictions.csv'
ens_df.to_csv(ens_csv, index=False)
print('Saved:', ens_csv)
ens_df.head()

## 6. Aggregate vs single-best baseline (Phase 2)

In [ ]:
summary = (ens_df.groupby('condition')
                  .agg(accuracy=('correct', 'mean'),
                       mean_trust=('trust', 'mean'),
                       mean_insertion=('insertion_auc', 'mean'),
                       n=('id_code', 'count'))).round(4).reset_index()

baseline = (stress_df[stress_df['model'] == best_clean]
             .assign(condition=lambda d: np.where(d['degradation'] == 'clean', 'clean',
                                                  d['degradation'] + '-' + d['level']))
             [['condition', 'accuracy', 'f1_macro']]
             .rename(columns={'accuracy': 'baseline_acc', 'f1_macro': 'baseline_f1'}))
summary = summary.merge(baseline, on='condition', how='left')
summary_path = PHASE_DIR / 'metrics' / 'ensemble_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)
summary

## 7. Plots

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Trust score distribution by condition
fig, ax = plt.subplots(figsize=(7, 4))
for cond in ens_df['condition'].unique():
    ax.hist(ens_df.loc[ens_df['condition'] == cond, 'trust'],
            bins=15, alpha=0.5, label=cond)
ax.set_title('Clinical Trust Score distribution by condition')
ax.set_xlabel('trust score'); ax.set_ylabel('count'); ax.legend()
plt.tight_layout()
out = PHASE_DIR / 'plots' / 'trust_score_distribution.png'
plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)

# Confusion matrix on the clean condition (most directly comparable to Phase 2)
clean = ens_df[ens_df['condition'] == 'clean']
if not clean.empty:
    cm = confusion_matrix(clean['true_label'], clean['pred'], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title('Confusion matrix — ensemble pipeline (clean)')
    fig.colorbar(im)
    plt.tight_layout()
    out = PHASE_DIR / 'plots' / 'confusion_matrix.png'
    plt.savefig(out, dpi=150); plt.show(); print('Saved:', out)

## 8. Qualitative figure of one routed example

In [ ]:
demo = ens_df.iloc[0]
src = APTOS_IMAGES / f"{demo['id_code']}.png" if demo['condition'] == 'clean' \
      else DEGRADED_DIR / demo['condition'] / 'high' / f"{demo['id_code']}.png"
img = Image.open(src).convert('RGB')
out = pipeline(img,
               kind_hint=None if demo['condition'] == 'clean' else demo['condition'],
               level_hint=None if demo['condition'] == 'clean' else 'high',
               id_hint=demo['id_code'])

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(img.resize((IMAGE_SIZE, IMAGE_SIZE))); axes[0].set_title(f'input ({demo["condition"]})')
axes[1].imshow(out['enhanced_img'].resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[1].set_title(f"after {out['route']['enhancement']}")
axes[2].imshow(out['enhanced_img'].resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[2].imshow(out['heatmap'], cmap='jet', alpha=0.45)
axes[2].set_title(f"{out['route']['xai']} — pred={out['pred']} true={demo['true_label']}")
for ax in axes: ax.axis('off')
fig.suptitle(f"Quality={out['quality_pred']} | Model={out['route']['model']} | trust={out['trust']:.2f}")
plt.tight_layout()
save = PHASE_DIR / 'samples' / f"route_{demo['id_code']}_{demo['condition']}.png"
plt.savefig(save, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', save)

---
**Phase 5 complete. The full project is now reproducible end-to-end.**

### Where everything lives on Drive
```
Thesis/
├── APTOS.zip / EYEQ.zip                # input
├── checkpoints/                         # *.pt for all classifiers + quality model
└── results/
    ├── phase1_data_engineering/
    │   ├── metrics/   plots/   samples/   logs/
    ├── phase2_model_benchmarking/
    ├── phase3_xai_benchmark/
    ├── phase4_genai_enhancement/
    └── phase5_quality_ensemble/
```
Each phase folder is **self-contained** — you can show a single phase to your professor without the others.